In [3]:
import polars as pl

In [5]:
from pathlib import Path

folder = Path("C:/Users/tobi/Documents/Machine-Learning/ps/project/sign_lang_train/sign_lang_train")

first_csv = next(folder.glob("*.csv"), None)



In [7]:
df = pl.read_csv(first_csv)

In [26]:
import polars as pl

df = pl.read_csv(
    first_csv,
    has_header=False,
    new_columns=["label", "image_path"]
)

print(df)

shape: (9_680, 2)
┌───────┬──────────────────────┐
│ label ┆ image_path           │
│ ---   ┆ ---                  │
│ str   ┆ str                  │
╞═══════╪══════════════════════╡
│ l     ┆ EOOYFWMXLNQZDRDC.jpg │
│ l     ┆ FUUDFATMEYSFGFAV.jpg │
│ l     ┆ LPMYBDJUNIHWWNYQ.jpg │
│ l     ┆ AWDPGQDXTGBMXDMV.jpg │
│ l     ┆ CTIMIWDGHBMIAIZJ.jpg │
│ …     ┆ …                    │
│ u     ┆ BMBCEPUITXNODAIT.jpg │
│ u     ┆ QYELBCJUQTMSPXOB.jpg │
│ u     ┆ IPTJYOKZAIGZUFWY.jpg │
│ u     ┆ LWUOUFCHUGHVEURN.jpg │
│ u     ┆ YFOBLENBWLJGINEG.jpg │
└───────┴──────────────────────┘


In [36]:
labels = df.select("label").unique("label").sort("label")
print([label[0] for label in labels.rows()])

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [39]:
counts = (
    df.group_by("label")
    .len()
    .rename({"len": "count"})
    .sort("count")
)

print(counts)

shape: (36, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ t     ┆ 104   │
│ f     ┆ 112   │
│ r     ┆ 112   │
│ 2     ┆ 112   │
│ a     ┆ 112   │
│ …     ┆ …     │
│ p     ┆ 560   │
│ 6     ┆ 560   │
│ 0     ┆ 560   │
│ g     ┆ 560   │
│ 4     ┆ 560   │
└───────┴───────┘


In [40]:
for row in counts.iter_rows(named=True):
    print(row)

{'label': 't', 'count': 104}
{'label': 'f', 'count': 112}
{'label': 'r', 'count': 112}
{'label': '2', 'count': 112}
{'label': 'a', 'count': 112}
{'label': 'e', 'count': 112}
{'label': '7', 'count': 112}
{'label': 'q', 'count': 112}
{'label': 'o', 'count': 112}
{'label': 'x', 'count': 112}
{'label': '3', 'count': 112}
{'label': '1', 'count': 112}
{'label': 'm', 'count': 112}
{'label': '5', 'count': 112}
{'label': 'n', 'count': 112}
{'label': 'h', 'count': 112}
{'label': 'w', 'count': 112}
{'label': 'd', 'count': 168}
{'label': 'k', 'count': 168}
{'label': '8', 'count': 168}
{'label': 'b', 'count': 280}
{'label': 'j', 'count': 280}
{'label': 'y', 'count': 280}
{'label': 's', 'count': 280}
{'label': 'i', 'count': 280}
{'label': 'v', 'count': 280}
{'label': 'c', 'count': 560}
{'label': 'l', 'count': 560}
{'label': 'z', 'count': 560}
{'label': '9', 'count': 560}
{'label': 'u', 'count': 560}
{'label': 'p', 'count': 560}
{'label': '6', 'count': 560}
{'label': '0', 'count': 560}
{'label': 'g',

In [23]:
from torchvision import datasets, transforms

In [ ]:
# Define hyperparameters
LEARNING_RATE = 0.001
MOMENTUM = 0.9
NUM_EPOCHS = 1
HIDDEN_SIZE = 100
TRAIN_BATCH_SIZE = 64
TEST_BATCH_SIZE = 64
INPUT_SIZE = 784
OUTPUT_SIZE = 10

In [43]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [45]:
import torch
from torch.utils.data import random_split, Subset

n_total = df.height

n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

generator = torch.Generator().manual_seed(42)

indices = range(n_total)

train_indices, val_indices, test_indices = random_split(
    indices,
    [n_train, n_val, n_test],
    generator=generator
)